# 03 From messy line list to clean data — notebook

Jun has exported the post-typhoon notifications from the City Health Office:
`santa_luz_linelist_raw.csv`, 637 rows typed by many hands. In this notebook you turn it
into the clean, de-identified line list that every later chapter uses — one small,
checked step at a time.

> **Everything here is invented.** Santa Luz City, Typhoon Amihan and every name, phone
> number and address in the raw file are synthetic training data. The names and numbers
> only *look* real so that you can practise protecting them.

The plan: **load and look → protect identities → fix dates → fix places → fix codes →
fix ages → remove duplicates → validate → prove it.**

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## Step 1 · Load and look

The file starts with a one-line note, so we skip one row. We also ask pandas to read
**every column as text** (`dtype=str`) and to leave blank cells as empty strings
(`keep_default_na=False`). Why? If pandas guesses types on a messy file, it quietly
changes things — `0900...` phone numbers lose their leading zero, `N/A` becomes a
missing value, `46227` becomes a number. We want to see the mess exactly as Jun saw it,
then decide every change ourselves.

We look only at a few non-identifying columns.

In [ ]:
import hashlib
import hmac
import io
import secrets

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAW_PATH = "data/synthetic/ph/santa_luz_linelist_raw.csv"

# skiprows=1 skips the note line; dtype=str keeps every cell as text;
# keep_default_na=False keeps blank cells as "" instead of NaN
raw = pd.read_csv(RAW_PATH, skiprows=1, dtype=str, keep_default_na=False)

print(f"{raw.shape[0]} rows, {raw.shape[1]} columns")
print(list(raw.columns))
raw[["Date Reported", "Sex", "Age", "Barangay", "Disease", "Date of Onset"]].head(8)

## Step 2 · Privacy first: protect identities, then clean

Chapter 02 taught the rule: **identifiers leave the table before the analysis starts.**
We still need the name for one job — spotting the same patient notified twice — so we
first turn it into the **salted pseudonymous ID** from chapter 02: the same tidied name
always gives the same ID, but without the secret salt nobody can redo it or read the
name back. Then we drop the name, phone number, address, reporting unit and
free-text remarks (remarks can name *other* people, too).

In the same cell we standardise the column names to `lower_case_with_underscores`, so
they are easy to type.

> **Ask Tala:** "I have a pandas DataFrame `raw` with text columns `Patient Name`,
> `Contact No.`, `Address`, `Reporting Unit`, `Remarks` and others. Made-up example:
> `Patient Name = 'Juana X. Halimbawa'`. Using my function `pseudo_id(name, salt)`, add a
> `pseudo_id` column, drop those five columns, and change all column names to
> snake_case."

In [ ]:
SALT = secrets.token_hex(16)  # demo only: in real work, create once and lock it away (chapter 02)


def pseudo_id(name: str, salt: str) -> str:
    key = " ".join(name.upper().replace(".", " ").split())   # normalise the name
    digest = hmac.new(salt.encode(), key.encode(), hashlib.sha256).hexdigest()
    return "P-" + digest[:10]


df = raw.copy()
df.insert(0, "pseudo_id", df["Patient Name"].map(lambda n: pseudo_id(n, SALT)))
df = df.drop(columns=["Patient Name", "Contact No.", "Address", "Reporting Unit", "Remarks"])

# "Date of Onset" -> "date_of_onset", "Did Cleanup Work" -> "did_cleanup_work"
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))
print(list(df.columns))

# Check it: same name typed two ways -> same ID; different name -> different ID; no identifiers left
assert pseudo_id("Juana X. Halimbawa", SALT) == pseudo_id("JUANA X  HALIMBAWA", SALT)  # made-up name
assert pseudo_id("Juana X. Halimbawa", SALT) != pseudo_id("Juan X. Halimbawa", SALT)
for column in ["patient_name", "contact_no", "address", "reporting_unit", "remarks"]:
    assert column not in df.columns
print("Identifiers removed; IDs look like this:", df["pseudo_id"].iloc[0])

## Step 3 · Tala got it wrong: the silent date swap

Jun's dates come in many styles: `07/24/2026`, `7/24/2026`, `2026-07-24`, `24-Jul-26`,
`July 24, 2026`, and plain numbers like `46227`.

> **Ask Tala:** "My column `date_of_onset` has dates in mixed formats. Convert it to
> datetime."

Tala answers quickly: *"Use `pd.to_datetime(..., format='mixed', dayfirst=True)` — most
of the world writes the day first."* The code runs with no error. That is the danger.
We keep Tala's answer in separate variables so it cannot spoil our data, and we test it
with two questions a nurse would ask: **how many dates failed?** and **does anyone fall
ill after they consulted?**

In [ ]:
# Tala's first answer -- kept apart so it cannot spoil df
naive_onset = pd.to_datetime(df["date_of_onset"], format="mixed", dayfirst=True, errors="coerce")
naive_consult = pd.to_datetime(df["date_consulted"], format="mixed", dayfirst=True, errors="coerce")

print("Onset dates that became NaT (not a time):", naive_onset.isna().sum())
print("Rows with onset AFTER consultation:", (naive_onset > naive_consult).sum())

swapped = naive_onset > naive_consult
pd.DataFrame({"onset (typed)": df.loc[swapped, "date_of_onset"],
              "consulted (typed)": df.loc[swapped, "date_consulted"],
              "Tala read consulted as": naive_consult[swapped].dt.date}).head(6)

Two problems, both silent:

1. **22 onset dates became `NaT`** — the Excel serial numbers (`46227`). To pandas they
   are not dates at all.
2. **56 patients now "fell ill after they consulted".** With `dayfirst=True`,
   `8/1/2026` (1 August) was read as 8 January. Dates like `7/29/2026` were still read
   correctly (there is no month 29), so the mistake hides among good rows. Only 2 of
   the 56 are real problems in Jun's file; the other 54 are Tala's.

In the Philippines the slash style is month/day/year, and Jun confirmed that every
slash date in this export follows it. So we write **one rule per format** and refuse to
guess.

> **Ask Tala (second try):** "Write a function `parse_dates(s)` for a pandas Series of
> text dates. Formats: `M/D/YYYY` (month first, leading zeros optional), `YYYY-MM-DD`,
> `DD-Mon-YY` (years are 20YY), `Month D YYYY` with or without a comma, and 5-digit Excel
> serial numbers (days since 1899-12-30). Blank means missing. Parse each format
> explicitly; do not let pandas guess."

In [ ]:
def parse_dates(s: pd.Series) -> pd.Series:
    """Parse Jun's date styles one explicit format at a time."""
    s = s.str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    serial = s.str.fullmatch(r"\d{5}")  # Excel counts days from 30 Dec 1899
    out[serial] = pd.to_datetime(s[serial].astype(int), unit="D", origin="1899-12-30")
    for fmt in ["%m/%d/%Y", "%Y-%m-%d", "%d-%b-%y", "%B %d %Y"]:
        attempt = pd.to_datetime(s.str.replace(",", "", regex=False), format=fmt, errors="coerce")
        out = out.fillna(attempt)  # only fills dates that are still missing
    return out


DATE_COLUMNS = {"date_reported": "date_report", "date_of_onset": "date_onset",
                "date_consulted": "date_consult", "date_lab_confirmed": "date_lab_confirmed"}
for old, new in DATE_COLUMNS.items():
    text = df[old]
    parsed = parse_dates(text)
    not_understood = parsed.isna() & text.str.strip().ne("")
    assert not_understood.sum() == 0, text[not_understood].unique()
    df[new] = parsed
    print(f"{new:<20} blank: {text.str.strip().eq('').sum():>3}   parsed: {parsed.notna().sum()}")

**Check it.** First a tiny case we can check by eye: the same day written seven ways.
Then the same two questions as before.

In [ ]:
same_day = pd.Series(["46227", "7/24/2026", "07/24/2026", "2026-07-24",
                      "24-Jul-26", "July 24 2026", "July 24, 2026", ""])
result = parse_dates(same_day)
print(result.dt.date.tolist())
assert (result[:7] == pd.Timestamp("2026-07-24")).all()
assert pd.isna(result[7])

print("Onset dates that are NaT:", df["date_onset"].isna().sum())
print("Rows with onset AFTER consultation:", (df["date_onset"] > df["date_consult"]).sum())

From 56 impossible rows down to **2**. Those two are real typing errors in the raw file
(an onset typed one month late) — we deal with them in Step 7.

## Step 4 · One barangay, many spellings

`Ilog`, `Brgy. Ilog`, `ILOG`, `Bgy Ilog ` (with a trailing space), `Barangay Ilog`…
plus `Sto. Nino` for Santo Niño and a few typos. We build a **key**: remove accents,
lower-case, drop the `barangay/brgy/bgy` prefix, write `sto` as `santo`, turn hyphens and
dots into spaces, and fix three known typos. We make the same key from the official
barangay list and join on it, which also brings in `barangay_id` and `band`.

**Check it:** an `assert` stops the notebook if any row finds no barangay. Watch the
count, too: the spellings become 23 barangays, because the 24th, upland Bukid, had no
notifications at all. Remember that in chapter 04 — a barangay with zero cases still
belongs in the table.

In [ ]:
TYPOS = {"pobalcion": "poblacion", "dalampasig": "dalampasigan", "kawayan": "kawayanan"}


def barangay_key(s: pd.Series) -> pd.Series:
    """Spelling-proof key: 'Brgy. Sto. Niño ' and 'santo nino' both become 'santo nino'."""
    key = (s.str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")  # ñ -> n
           .str.lower().str.strip()
           .str.replace(r"^(barangay|brgy\.?|bgy\.?)\s+", "", regex=True)
           .str.replace(r"^sto\.?\s+", "santo ", regex=True)
           .str.replace(r"[-.]", " ", regex=True)
           .str.split().str.join(" "))
    return key.replace(TYPOS)


barangays = pd.read_csv("data/synthetic/ph/santa_luz_barangays.csv")
barangays["key"] = barangay_key(barangays["barangay"])

df = df.rename(columns={"barangay": "barangay_raw"})
df["key"] = barangay_key(df["barangay_raw"])
df = df.merge(barangays[["key", "barangay_id", "barangay", "band"]], on="key", how="left")

unmatched = df.loc[df["barangay_id"].isna(), "barangay_raw"].unique()
assert len(unmatched) == 0, unmatched  # Check it: every row found its barangay
print("Raw spellings:", df["barangay_raw"].nunique(), "-> official barangays:", df["barangay"].nunique())

# How many spellings did each barangay have? A chart Jun may want to show the encoders.
spellings = df.groupby("barangay")["barangay_raw"].nunique().sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(spellings.index, spellings.values, color="#0038A8")
ax.set_xlabel("Different spellings in Jun's export")
ax.set_title("One barangay, many spellings")
plt.tight_layout()
plt.show()

print(df.loc[df["barangay"] == "Santo Niño", "barangay_raw"].unique())

## Step 5 · Codes in Taglish: sex, disease, classification, yes/no, outcome, severity

Each field gets a small **lookup table** (a dictionary) or a rule. Anything not in the
table becomes missing, so the checks at the end of the cell catch any spelling we forgot.

| Field | Raw examples | Clean |
|---|---|---|
| Sex | `M`, `male`, `lalaki` / `F`, `Female`, `babae` | `M` / `F` |
| Disease | `Lepto`, `LEPTO`… / `DENGUE`, `Dengue Fever`, `DF` | `leptospirosis` / `dengue` |
| Classification | `lab confirmed`, `Conf` / `prob.` / `sus`, `Suspected` | `confirmed` / `probable` / `suspect` |
| Yes/no | `Y`, `yes`, `oo` / `N`, `no`, `hindi` / blank, `N/A` | `1` / `0` / missing |
| Outcome | `buhay`, `A` / `Namatay`, `Expired`, `D` | `alive` / `died` |

In [ ]:
sex = df["sex"].str.strip().str.lower()
df["sex"] = sex.map({"m": "M", "male": "M", "lalaki": "M", "f": "F", "female": "F", "babae": "F"})

disease = df["disease"].str.strip().str.lower()
df["disease"] = np.select([disease.str.startswith("lep"),
                           disease.str.startswith("deng") | disease.eq("df")],
                          ["leptospirosis", "dengue"], default=None)

cls = df["classification"].str.strip().str.lower()
df["case_class"] = np.select([cls.str.contains("conf"), cls.str.startswith("prob"),
                              cls.str.startswith("sus")],
                             ["confirmed", "probable", "suspect"], default=None)

YES_NO = {"y": 1, "yes": 1, "oo": 1, "n": 0, "no": 0, "hindi": 0}  # blank and "n/a" -> missing
for old, new in {"admitted": "hospitalised", "waded_in_floodwater": "waded_floodwater",
                 "did_cleanup_work": "cleanup_work"}.items():
    df[new] = df[old].str.strip().str.lower().map(YES_NO).astype("Int64")

outcome = df["outcome"].str.strip().str.lower()
df["outcome"] = outcome.map({"alive": "alive", "a": "alive", "buhay": "alive",
                             "died": "died", "d": "died", "namatay": "died", "expired": "died"})

sev = df["dengue_severity"].str.strip().str.lower()
df["dengue_severity"] = np.select(  # order matters: "without" also contains "with"
    [sev.str.contains("severe") | sev.eq("sd"),
     sev.str.contains("without") | sev.str.contains("w/o", regex=False) | sev.eq("no ws"),
     sev.str.contains("with") | sev.str.contains("w/", regex=False) | sev.eq("ws+")],
    ["severe", "no_warning", "warning_signs"], default=None)
df.loc[df["disease"] != "dengue", "dengue_severity"] = None

# Check: nothing fell through the lookup tables
for column in ["sex", "disease", "case_class", "outcome", "hospitalised"]:
    assert df[column].notna().all(), column
assert df.loc[df["disease"] == "dengue", "dengue_severity"].notna().all()
print(df["disease"].value_counts().to_dict(), df["sex"].value_counts().to_dict())

## Step 6 · Ages: "34 y/o", "8 mos" and one 150

Take the first whole number. If the text contains `mo`, it is months, so divide by 12
and keep the whole years (`8 mos` → 0, as on the case investigation form for infants).
An age above 110 is impossible, so it becomes missing — in real life you would call the
reporting unit. Then we put ages into the standard groups.

In [ ]:
def parse_age(s: pd.Series) -> pd.Series:
    """Completed years from text like '34', '34 y/o', '34yo', '8 mos', '8 months'."""
    text = s.str.strip().str.lower()
    number = text.str.extract(r"^(\d+)")[0].astype("Int64")
    years = number.where(~text.str.contains("mo"), number // 12)  # months -> years
    return years.where(years <= 110).astype("Int64")             # 150 -> missing


# Check on a case you can do in your head
print(parse_age(pd.Series(["34", "34 y/o", "34yo", "8 mos", "18 months", "150", ""])).tolist())

df["age"] = parse_age(df["age"])
df["age_group"] = pd.cut(df["age"], bins=[-1, 4, 14, 24, 44, 64, 200],
                         labels=["0-4", "5-14", "15-24", "25-44", "45-64", "65+"]).astype(object)
print("Missing ages:", df["age"].isna().sum(), "| oldest:", df["age"].max())

## Step 7 · Duplicates, impossible dates, and new IDs

**Exact copies** are easy: `df.duplicated()`. **Near-duplicates** are harder: the City
Health Office re-notified some patients with the name re-typed, another date style and
another barangay spelling. After our cleaning those rows *look* the same in the columns
that matter, so we de-duplicate on **(pseudo ID, disease, onset date)** and keep the first.

Then three last rules:

- **Impossible dates:** if onset is after consultation (or after the report date when
  consultation is blank), set onset to missing — and, in real life, phone the
  reporting unit.
- **Pseudonymous case IDs** `SLC-0001`, `SLC-0002`, … in file order.
- **Columns** in the agreed order. The pseudo ID, raw text and identifiers stay behind.

In [ ]:
print("Exact copies:", df.duplicated().sum())
duplicate = df.duplicated(subset=["pseudo_id", "disease", "date_onset"], keep="first")
print("Same person + disease + onset (exact and near):", duplicate.sum())
df = df[~duplicate].reset_index(drop=True)

later = df["date_consult"].fillna(df["date_report"])
impossible = df["date_onset"] > later
print("Onset after consultation/report -> set to missing:", impossible.sum())
df.loc[impossible, "date_onset"] = pd.NaT

df["case_id"] = [f"SLC-{i:04d}" for i in range(1, len(df) + 1)]

COLUMNS = ["case_id", "disease", "case_class", "date_onset", "date_consult", "date_report",
           "date_lab_confirmed", "age", "age_group", "sex", "barangay_id", "barangay", "band",
           "hospitalised", "outcome", "dengue_severity", "waded_floodwater", "cleanup_work"]
clean = df[COLUMNS]
print(clean.shape)
clean.head()

## Step 8 · Validation rules as assertions

A **validation rule** is a sentence about the data that must always be true, written as
an `assert`. If a future export breaks a rule, the notebook stops loudly instead of
producing a wrong report quietly. Read each line aloud — they are epidemiology, not
just code.

In [ ]:
assert clean["case_id"].is_unique
assert clean["date_report"].between(pd.Timestamp("2026-07-20"), pd.Timestamp("2026-09-30")).all()
both = clean.dropna(subset=["date_onset", "date_consult"])
assert (both["date_onset"] <= both["date_consult"]).all()          # ill before consulting
assert (clean["date_consult"] <= clean["date_report"]).where(clean["date_consult"].notna(), True).all()
assert clean["age"].dropna().between(0, 110).all()
assert set(clean["sex"]) == {"M", "F"}
confirmed = clean["case_class"] == "confirmed"
assert clean.loc[confirmed, "date_lab_confirmed"].notna().all()     # confirmed = lab result
assert clean.loc[clean["disease"] == "dengue", "waded_floodwater"].isna().all()
print("All validation rules pass.")
print(clean["disease"].value_counts().to_dict())
print("Missing: onset", clean["date_onset"].isna().sum(), "| consult", clean["date_consult"].isna().sum(),
      "| age", clean["age"].isna().sum())

## Step 9 · Prove it

Doc Mon's question: *"Is yours the same as the official clean file?"* We save our table
the way rule C8 says (`to_csv(index=False)`, here into memory), read it back, and let
pandas compare it with `santa_luz_linelist.csv` cell by cell. `assert_frame_equal`
prints nothing when the tables are identical and explains the first difference when
they are not.

In [ ]:
csv_text = clean.to_csv(index=False, lineterminator="\n")  # what we would save
ours = pd.read_csv(io.StringIO(csv_text))
official = pd.read_csv("data/synthetic/ph/santa_luz_linelist.csv")

pd.testing.assert_frame_equal(ours, official)
print(f"Identical to santa_luz_linelist.csv: {ours.shape[0]} rows x {ours.shape[1]} columns")

## Your turn

1. **Filipino month names.** A new encoder types `Hulyo 24, 2026`. In a new cell, run
   `parse_dates(pd.Series(["Hulyo 24, 2026"]))`. What do you get, and does the `assert`
   in Step 3 protect you? *Hint: replace `Hulyo` with `July` (and `Agosto` with `August`)
   using `.str.replace` before parsing, then re-run the seven-ways check.*
2. **A new spelling.** What does `barangay_key(pd.Series(["Brgy Tabing  Ilog"]))` return?
   Does it match `tabing ilog`? Now try `"Tabing-ilog (riverside)"`. *Hint: the key
   cannot fix everything. Find the row with `.str.contains("Tabing")`, and add a rule
   only when you know what the encoder meant.*
3. **Count the work.** How many rows did each step change? *Hint: compare
   `len(raw)` with `len(clean)`, and count `naive_onset.isna()` against
   `df["date_onset"].isna()`. Write the numbers into a three-line note for Jun.*